# 03 · Detection Tuning **v3**

## v2'nin ÇÖKEN modeli
`jaccard ≈ recall²` **yanlış çıktı**. Gerçek tam-koşu:

| konfig | recall | **gerçek ham jaccard** |
|---|---|---|
| baseline `(1,2,2)\|(3,11,11)` — 313/kare | 0.784 | **0.680** |
| bombardıman `(0.5,1,1)\|(1,5,5)` — 3413/kare | 0.988 | **0.303** |

`44b6_0113de3b`: recall **1.0** ama **TP=13 / FN=37** (baseline'da TP=47!).
→ Node'lar bulunuyor ama **birbirine bağlanmıyor**: 1414 node/kare'de Hungarian, GT'ye
eşleşen node'u komşu **sahte** tespitlere atıyor.

> **Linking, yoğunluk arttıkça çöküyor.** "Linking sorun değil" tespitim yalnızca
> normal yoğunlukta geçerliymiş. Üstelik bu, ceza **öncesi** ham skor.

## v3'te düzeltilenler
1. **Gerçek `T_true`** kullanılıyor: `.geff` → `attrs["geff"]["extra"]["estimated_number_of_nodes"]`.
   (25.755 / 32.795 / 6.362 / 69.800 — dataset başına 11× değişiyor, Otsu bunu doğal olarak tutturuyor.)
2. **Sıralama `est_adj`'e göre** (recall² × gerçek ceza) — recall'a göre değil.
   Bombardıman konfigleri böylece otomatik eleniyor.
3. **Tam doğrulama elle seçilmiş adaylarda** — sweep'in tepesine körlemesine güvenmiyoruz.
4. Görsel teşhis **öne alındı** (hızlı; uzun koşuyu beklemeden görülsün).

## Hedef
Yoğunluk zaten optimal (oran 0.94–1.06, ceza ≈ 0). Kovalayacağımız şey:
**aynı bütçede daha isabetli tespit.**

## 0 · Kurulum

In [ ]:
import sys, subprocess, os, time, itertools
def _scan(base):
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.endswith((".zarr",".geff")) and d!="competitions"]
        if root.count(os.sep) > 9: dirs[:]=[]; continue
        yield root, files
def ensure_zarr():
    try:
        import zarr; return zarr
    except ImportError: pass
    for root, files in _scan("/kaggle/input"):
        if os.path.basename(root)=="zarr" and "__init__.py" in files:
            p=os.path.dirname(root); sys.path.insert(0,p)
            try:
                import zarr; print("zarr <- sys.path:",p); return zarr
            except ImportError: sys.path.pop(0)
    subprocess.run([sys.executable,"-m","pip","install","-q","zarr"],check=False)
    import zarr; return zarr
zarr=ensure_zarr(); print("zarr:",zarr.__version__)

import numpy as np, pandas as pd
from pathlib import Path
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

SCALE=(1.625,0.40625,0.40625); S=np.array(SCALE,dtype=np.float32)
MATCH_UM=7.0; LINK_MAX_UM=8.0; ALPHA=0.1
FIG=Path("/kaggle/working/figures"); FIG.mkdir(parents=True,exist_ok=True)

INPUT=Path("/kaggle/input")
def find_root():
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/"train").is_dir() and (b/"test").is_dir(): return b
        except Exception: pass
        if d<4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr",".geff")): st.append((c,d+1))
ROOT=find_root(); TRAIN=ROOT/"train"; TEST=ROOT/"test"
test_names=sorted(p.stem for p in TEST.glob("*.zarr"))

def open_image(zp):
    n=zarr.open(str(zp),mode="r"); a=dict(n.attrs)
    ms=a.get("multiscales") or (a.get("ome") or {}).get("multiscales")
    if ms: return n[ms[0]["datasets"][0]["path"]]
    return n["0"] if "0" in list(n.keys()) else n

def load_geff(gp):
    g=zarr.open(str(gp),mode="r"); nodes=g["nodes"]
    ids=np.asarray(nodes["ids"]); props={}
    for pn in list(nodes["props"].keys()):
        try: props[pn]=np.asarray(nodes["props"][pn]["values"])
        except Exception: pass
    d={"id":ids}
    for k in ("t","z","y","x"):
        if k in props: d[k]=props[k]
    # RESMI hedef node sayisi (metrigin n_total'i)
    est=dict(g.attrs).get("geff",{}).get("extra",{}).get("estimated_number_of_nodes")
    return pd.DataFrame(d), np.asarray(g["edges"]["ids"]), est
print("test:",test_names)

## 1 · Kareleri + **resmî T_true**'yu yükle

In [ ]:
TARGET_GT=40; MAX_FR=12
FRAMES=[]; GT_ALL={}; T_TRUE={}
t0=time.time()
for nm in test_names:
    gdf,ge,est=load_geff(TRAIN/(nm+".geff")); GT_ALL[nm]=(gdf,ge); T_TRUE[nm]=est
    arr=open_image(TEST/(nm+".zarr"))
    cnt=gdf.groupby("t").size().sort_values(ascending=False)
    picks=[]; tot=0
    for t,c in cnt.items():
        picks.append(int(t)); tot+=int(c)
        if tot>=TARGET_GT or len(picks)>=MAX_FR: break
    for t in picks:
        v=np.asarray(arr[t]).astype(np.float32)
        gp=gdf[gdf.t==t][["z","y","x"]].values.astype(np.float32)
        FRAMES.append((nm,t,v,gp))
print(f"{len(FRAMES)} kare, {time.time()-t0:.0f}s, RAM ~{sum(f[2].nbytes for f in FRAMES)/1e6:.0f} MB\n")
print("RESMI T_true (estimated_number_of_nodes):")
for nm in test_names:
    fr=[f for f in FRAMES if f[0]==nm]
    print(f"  {nm}: T_true={T_TRUE[nm]:>6} ({T_TRUE[nm]/100:>5.0f}/kare) | "
          f"{len(fr):2d} kare, {sum(len(f[3]) for f in fr):3d} GT node")

## 2 · Detection + değerlendirme (gerçek cezalı)

In [ ]:
def detect(v, sigma, foot, thr_mode):
    sm=ndi.gaussian_filter(v, sigma=sigma)
    if thr_mode=="otsu": thr=threshold_otsu(sm)
    elif thr_mode.startswith("otsu*"): thr=threshold_otsu(sm)*float(thr_mode.split("*")[1])
    elif thr_mode.startswith("p"): thr=np.percentile(sm, float(thr_mode[1:]))
    mx=ndi.maximum_filter(sm, size=foot)
    peaks=(sm==mx)&(sm>thr)
    lbl,n=ndi.label(peaks)
    if n==0: return np.zeros((0,3),np.float32)
    return np.asarray(ndi.center_of_mass(sm,lbl,np.arange(1,n+1)),dtype=np.float32)

def penalty(T_pred, T_true):
    # metrics.py: max(0, J * (1 - 0.1*(T_pred - T_true)/T_true))
    return max(0.0, 1 - ALPHA*(T_pred - T_true)/T_true)
print("ok")

## 3 · 🔍 Kırık dataset teşhisi — kaçırılan GT'nin etrafında ne var?
`44b6_0b24845f`: **doğru sayıda** (≈T_true) tespit ama recall 0.33 → *doğru sayı, yanlış nesneler*.
**Gerçekçi** konfigle (bombardıman değil) bakıyoruz.

In [ ]:
BAD="44b6_0b24845f"; DIAG=((1,2,2),(3,11,11),"otsu")
shown=0
fig,axes=plt.subplots(2,3,figsize=(15,10)); axes=axes.ravel()
for nm,t,v,gp in FRAMES:
    if nm!=BAD or shown>=6: continue
    c=detect(v,*DIAG)
    D=cdist(gp*S,c*S) if len(c) else np.full((len(gp),1),1e9)
    for i,p in enumerate(gp):
        if shown>=6: break
        if D[i].min()<=MATCH_UM: continue          # sadece KACIRILANLAR
        z,y,x=int(round(p[0])),int(round(p[1])),int(round(p[2])); R=40
        y0,y1=max(0,y-R),min(v.shape[1],y+R); x0,x1=max(0,x-R),min(v.shape[2],x+R)
        a=axes[shown]; a.imshow(v[z,y0:y1,x0:x1],cmap="gray")
        a.plot(x-x0,y-y0,"rx",ms=16,mew=3)
        if len(c):
            for q in c[np.abs(c[:,0]-z)<=2]:
                if y0<=q[1]<y1 and x0<=q[2]<x1: a.plot(q[2]-x0,q[1]-y0,"g+",ms=10,mew=2)
        a.set_title(f"t={t} z={z} | en yakın tespit {D[i].min():.1f}µm | GT parlaklık={v[z,y,x]:.0f}",fontsize=8)
        a.axis("off"); shown+=1
for j in range(shown,6): axes[j].axis("off")
plt.suptitle(f"{BAD} — kırmızı ✗ = kaçırılan GT, yeşil + = tespitlerimiz  (konfig {DIAG[0]}|{DIAG[1]})")
plt.tight_layout(); plt.savefig(FIG/"D12_missed_gt.png",dpi=120,bbox_inches="tight"); plt.show()
print("kacirilan GT gosterildi:",shown)

## 4 · Sweep — **gerçek T_true ile sıralama**
`est_adj = ort( recall_ds² × ceza(T_pred_ds, T_true_ds) )`.
Grid makul yoğunluklara odaklı (bombardıman bölgesi zaten elendi); eşiği **iki yöne** de tarıyoruz.

In [ ]:
SIGMAS=[(0.5,1,1), (0.75,1.5,1.5), (1,2,2)]
FOOTS =[(3,7,7), (3,11,11), (5,15,15)]
THRS  =["otsu*0.8", "otsu", "otsu*1.2"]

rows=[]; t0=time.time()
for sg,ft,th in itertools.product(SIGMAS,FOOTS,THRS):
    per={nm:{"m":0,"g":0,"dens":[]} for nm in test_names}
    for nm,t,v,gp in FRAMES:
        c=detect(v,sg,ft,th)
        ok=int((cdist(gp*S,c*S).min(1)<=MATCH_UM).sum()) if (len(c) and len(gp)) else 0
        per[nm]["m"]+=ok; per[nm]["g"]+=len(gp); per[nm]["dens"].append(len(c))
    M=sum(p["m"] for p in per.values()); G=sum(p["g"] for p in per.values())
    adj=[]
    r=dict(sigma=str(sg),foot=str(ft),thr=th,recall=round(M/max(G,1),4))
    for nm in test_names:
        p=per[nm]; rec=p["m"]/max(p["g"],1); Tp=float(np.mean(p["dens"]))*100
        a=rec*rec*penalty(Tp,T_TRUE[nm]); adj.append(a)
        r[nm[:9]]=round(rec,2); r[nm[:9]+"_x"]=round(Tp/T_TRUE[nm],2)
    r["est_adj"]=round(float(np.mean(adj)),4)
    rows.append(r)
    print(f"{str(sg):16s} {str(ft):11s} {th:9s} recall={r['recall']:.3f} est_adj={r['est_adj']:.3f}")
res=pd.DataFrame(rows).sort_values("est_adj",ascending=False)
print(f"\n{len(rows)} konfig, {time.time()-t0:.0f}s\n")
print("=== EN IYI 12 (est_adj — gercek T_true cezasiyla) ===")
print(res.head(12).to_string(index=False))
print("\n(_x kolonlari = T_pred/T_true orani; 1.0 ideal)")

### 4a · Görselleştirme

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].scatter(res.recall,res.est_adj,s=60,c="#4C78A8")
for r in res.head(3).itertuples(): ax[0].annotate(f"{r.sigma}\n{r.foot}|{r.thr}",(r.recall,r.est_adj),fontsize=6)
ax[0].set_xlabel("havuzlanmış recall"); ax[0].set_ylabel("est_adj (cezalı)")
ax[0].set_title("recall yüksek ≠ skor yüksek")
b=res.head(10)
ax[1].barh(range(len(b)), b.est_adj[::-1])
ax[1].set_yticks(range(len(b)))
ax[1].set_yticklabels([f"{r.sigma}|{r.foot}|{r.thr}" for r in b.itertuples()][::-1],fontsize=7)
ax[1].set_xlabel("est_adj"); ax[1].set_title("en iyi 10")
plt.tight_layout(); plt.savefig(FIG/"D13_sweep_v3.png",dpi=120,bbox_inches="tight"); plt.show()

## 5 · **Gerçek** doğrulama — elle seçilmiş adaylar
Sweep'in tepesine körlemesine güvenmiyoruz (v2'de bu 22 dk yaktı). Makul adaylar + kontrol.
Skor artık **cezalı** (`adj_jaccard`), yani gerçek leaderboard metriği.

In [ ]:
def link_pairs(A,B):
    if len(A)==0 or len(B)==0: return []
    D=cdist(A*S,B*S); cost=np.where(D<=LINK_MAX_UM,D,1e6)
    r,c=linear_sum_assignment(cost)
    return [(int(i),int(j)) for i,j in zip(r,c) if D[i,j]<=LINK_MAX_UM]

def track_eval(nm, sigma, foot, thr):
    arr=open_image(TEST/(nm+".zarr")); T=arr.shape[0]
    cents=[detect(np.asarray(arr[t]).astype(np.float32),sigma,foot,thr) for t in range(T)]
    nodes=[]; edges=[]; off=[]; nid=1
    for t,c in enumerate(cents):
        off.append(nid)
        for p in c:
            nodes.append((nid,t,int(round(p[0])),int(round(p[1])),int(round(p[2])))); nid+=1
    for t in range(T-1):
        for i,j in link_pairs(cents[t],cents[t+1]): edges.append((off[t]+i,off[t+1]+j))
    gdf,ge = GT_ALL[nm][0], GT_ALL[nm][1]
    pn=pd.DataFrame(nodes,columns=["node_id","t","z","y","x"]); gmap={}
    for t,g in gdf.groupby("t"):
        p=pn[pn.t==int(t)]
        if len(p)==0 or len(g)==0: continue
        D=cdist(p[["z","y","x"]].values*S, g[["z","y","x"]].values*S)
        cost=np.where(D<=MATCH_UM,D,1e6); r,c=linear_sum_assignment(cost)
        pid=p["node_id"].values; gid=g["id"].values
        for i,j in zip(r,c):
            if D[i,j]<=MATCH_UM: gmap[int(pid[i])]=int(gid[j])
    gtset=set((int(u),int(v)) for u,v in ge); TP=0;FP=0;cov=set()
    for u,v in edges:
        gu=gmap.get(u); gv=gmap.get(v)
        if gu is None or gv is None: continue
        if (gu,gv) in gtset: TP+=1; cov.add((gu,gv))
        else: FP+=1
    FN=len(gtset)-len(cov); J=TP/max(TP+FP+FN,1)
    pen=penalty(len(nodes), T_TRUE[nm])
    return dict(dataset=nm, jaccard=round(J,4), adj_jaccard=round(J*pen,4),
                recall=round(len(set(gmap.values()))/max(len(gdf),1),3),
                TP=TP,FN=FN, nodes=len(nodes), x_true=round(len(nodes)/T_TRUE[nm],2))

CANDIDATES=[((0.5,1,1),(3,7,7),"otsu"),
            ((0.5,1,1),(3,11,11),"otsu"),
            ((1,2,2),(3,11,11),"otsu")]     # <- kontrol = mevcut baseline
RUN_FULL=True
if RUN_FULL:
    summary=[]
    for sg,ft,th in CANDIDATES:
        print(f"\n=== {sg} | {ft} | {th} ===")
        out=[]; t0=time.time()
        for nm in test_names:
            r=track_eval(nm,sg,ft,th); out.append(r); print("  ",r)
        d=pd.DataFrame(out)
        summary.append(dict(konfig=f"{sg}|{ft}|{th}",
                            adj_jaccard=round(d.adj_jaccard.mean(),4),
                            ham_jaccard=round(d.jaccard.mean(),4),
                            recall=round(d.recall.mean(),3),
                            x_true=round(d.x_true.mean(),2), sn=int(time.time()-t0)))
        print("  ORT adj_jaccard=%.4f | ham=%.4f | recall=%.3f | T_pred/T_true=%.2f (%ds)"%(
              d.adj_jaccard.mean(), d.jaccard.mean(), d.recall.mean(), d.x_true.mean(), time.time()-t0))
    print("\n=== KARSILASTIRMA ===")
    print(pd.DataFrame(summary).sort_values("adj_jaccard",ascending=False).to_string(index=False))
    print("\n(v1 baseline referans: ham jaccard 0.680)")

## 6 · Karar

- [ ] Kazanan konfig = **…** → adj_jaccard **…** (baseline 0.68)
- [ ] `44b6_0b24845f` görselinde ne var? (kaçan GT'nin yanındaki parlak yapı nedir?)
- [ ] `T_pred/T_true` ≈ 1.0 kaldı mı?

Kazananı `02_baseline.ipynb`'e taşı → submit.
Sonraki lever: **linking kalitesi** (yoğunlukla bozuluyor) ve **bölünme** (%10).